# MoodMentor Milestone 2: GoEmotions GPU run

This notebook runs the existing repository workflow on a Colab GPU. It does not claim near-100% performance: GoEmotions is noisy, multi-label, and human-agreement limited. Validation selects checkpoints and thresholds; the held-out test split is scored only afterward.

**Upload three zip files when prompted:** the repository source, the attached `archive (1)/data`, and the attached `models/goemotions-distilbert` checkpoint. Do not upload secrets or generated artifacts.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

BASE = Path('/content/moodmentor-colab')
BASE.mkdir(parents=True, exist_ok=True)
def upload_and_extract(message, destination):
    print(message)
    uploaded = files.upload()
    name = next(iter(uploaded))
    archive = Path('/content') / name
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(destination)

upload_and_extract('Upload a zip of the repository source.', BASE)
upload_and_extract('Upload a zip containing archive (1)/data.', BASE / 'attached')
upload_and_extract('Upload a zip containing models/goemotions-distilbert.', BASE / 'attached')

repo_candidates = list(BASE.rglob('scripts/train_goemotions_model.py'))
if not repo_candidates:
    raise FileNotFoundError('Repository zip must contain scripts/train_goemotions_model.py')
REPO = repo_candidates[0].parents[1]
DATA = next(path.parent for path in (BASE / 'attached').rglob('emotions.txt'))
MODEL = next(path.parent for path in (BASE / 'attached').rglob('model.safetensors'))
print({'repo': str(REPO), 'data': str(DATA), 'model': str(MODEL)})

## Install and verify GPU

In [ ]:
%cd $REPO
%pip install -q -e '.[ml,dev]'
import torch
labels = [line.strip() for line in (DATA / 'emotions.txt').read_text().splitlines() if line.strip()]
assert len(labels) == 28 and len(set(labels)) == 28
print({'cuda_available': torch.cuda.is_available(), 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU', 'labels': len(labels)})
if not torch.cuda.is_available():
    raise RuntimeError('Colab GPU is not active. Choose Runtime > Change runtime type > T4 GPU and rerun.')

## Fine-tune DistilBERT and BERT

Both runs use identical splits, seed, threshold policy, and full-data scope.

In [ ]:
RESULTS = Path('/content/moodmentor-results')
RESULTS.mkdir(exist_ok=True)
!python -m scripts.train_goemotions_model --data-dir "$DATA" --base-model "$MODEL" --epochs 3 --batch-size 16 --learning-rate 2e-5 --max-length 64 --threshold 0.5 --positive-weight-cap 8 --seed 42 --output-dir "$RESULTS/attached-distilbert-full" --metrics-output "$RESULTS/attached-distilbert-full.json"
!python -m scripts.train_goemotions_model --data-dir "$DATA" --base-model bert-base-uncased --epochs 3 --batch-size 16 --learning-rate 2e-5 --max-length 64 --threshold 0.5 --positive-weight-cap 8 --seed 42 --output-dir "$RESULTS/bert-full" --metrics-output "$RESULTS/bert-full.json"

## Validation-only calibration and held-out evaluation

In [ ]:
!python -m scripts.calibrate_goemotions --data-dir "$DATA" --model "$RESULTS/attached-distilbert-full" --batch-size 16 --output "$RESULTS/attached-distilbert-calibration.json"
!python -m scripts.calibrate_goemotions --data-dir "$DATA" --model "$RESULTS/bert-full" --batch-size 16 --output "$RESULTS/bert-calibration.json"
!python -m scripts.compare_training_runs "$RESULTS/attached-distilbert-full.json" "$RESULTS/bert-full.json" --output "$RESULTS/model-comparison.json"

## Apply calibration hot-fix without retraining

If the training cells already completed before the CUDA calibration fix, upload the rebuilt `moodmentor-project-safe.zip` once here. This overwrites only the repository scripts in `/content/moodmentor-colab`; the trained model outputs under `/content/moodmentor-results` are preserved.

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

print('Upload the rebuilt moodmentor-project-safe.zip only.')
uploaded = files.upload()
archive = Path('/content') / next(iter(uploaded))
with zipfile.ZipFile(archive) as handle:
    handle.extractall('/content/moodmentor-colab')
print('Calibration script updated; existing trained models were preserved.')

## Select the strongest honest result

This selects the model using validation-calibrated macro-F1 only, then reports its untouched calibrated test score. The test score is not used to choose the winner.

In [ ]:
import json

reports = {
    'attached-distilbert': json.loads((RESULTS / 'attached-distilbert-calibration.json').read_text()),
    'bert': json.loads((RESULTS / 'bert-calibration.json').read_text()),
}
ranking = sorted(
    key=lambda row: row[1],
    reverse=True,
)
print('Selected by validation-calibrated macro-F1:')
for name, validation_f1, heldout_f1 in ranking:
    print(f'{name}: validation={validation_f1:.4f}, untouched_test={heldout_f1:.4f}')
print('Do not describe the selected score as near-100%; report the exact held-out value.')

## Review rules

- A run with a non-null `limit` is a smoke test, not milestone evidence.
- Do not call the `0.82` target passed unless a fresh validation artifact reaches it.
- Report fixed and validation-calibrated held-out macro-F1, micro-F1, precision, recall, exact match, and per-label metrics.
- Do not claim the `0.90` SemEval target without the mentor-approved local export.
- Download only aggregate JSON reports; do not commit model weights or raw data.